# California House Price Prediction – Linear Model
**Steps:** 1) Load data → 2) Explore → 3) Preprocess → 4) Train a linear model → 5) Check overfitting → 6) Small UI to predict a price

> Target `MedHouseVal` is in units of **$100,000**.
> Install requirements if needed: `pip install numpy pandas matplotlib scikit-learn ipywidgets`

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_validate, learning_curve
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

RANDOM_STATE = 42
plt.rcParams["figure.figsize"] = (7, 4)

## 1. Load the data

In [ ]:
data = fetch_california_housing(as_frame=True)
df = data.frame.copy()
print(df.shape)
df.head()

## 2. Quick exploration

In [ ]:
print("Missing values per column:")
print(df.isna().sum())
df.describe().T

In [ ]:
df.hist(bins=40, figsize=(12, 8))
plt.tight_layout()
plt.show()

# Correlation of each feature with the target
df.corr()["MedHouseVal"].sort_values(ascending=False)

Observations to keep in mind:
- Some features (`AveRooms`, `AveBedrms`, `AveOccup`, `Population`) have **extreme outliers**.
- The target is **capped at 5.0** (= $500,000), so many houses share the max value. Those rows are censored, which hurts a linear model.

## 3. Preprocessing
- Remove rows where the target is capped (5.0).
- Split into train / test **before** learning anything from the data (avoids data leakage).
- Inside a `Pipeline`: clip outliers (percentiles learned on the training set only) → standardize features.

In [ ]:
# Remove censored target values
before = len(df)
df = df[df["MedHouseVal"] < 5.0].reset_index(drop=True)
print(f"Removed {before - len(df)} rows with capped target")

FEATURES = list(data.feature_names)   # the 8 input variables
X = df[FEATURES]
y = df["MedHouseVal"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)
print(X_train.shape, X_test.shape)

In [ ]:
class Winsorizer(BaseEstimator, TransformerMixin):
    """Clip each feature to [lower, upper] percentiles learned on the training data."""
    def __init__(self, lower=0.01, upper=0.99):
        self.lower = lower
        self.upper = upper

    def fit(self, X, y=None):
        X = pd.DataFrame(X)
        self.low_ = X.quantile(self.lower).values
        self.high_ = X.quantile(self.upper).values
        return self

    def transform(self, X):
        return np.clip(np.asarray(X), self.low_, self.high_)

def make_model(regressor=None):
    return Pipeline([
        ("clip", Winsorizer(0.01, 0.99)),
        ("scale", StandardScaler()),
        ("reg", regressor if regressor is not None else LinearRegression()),
    ])

## 4. Train a simple linear model

In [ ]:
model = make_model(LinearRegression())
model.fit(X_train, y_train)

coefs = pd.Series(model.named_steps["reg"].coef_, index=FEATURES).sort_values()
print("Intercept:", round(model.named_steps["reg"].intercept_, 4))
coefs.plot(kind="barh", title="Standardized coefficients")
plt.show()

In [ ]:
def evaluate(model, X, y):
    pred = model.predict(X)
    return {
        "R2": r2_score(y, pred),
        "RMSE": np.sqrt(mean_squared_error(y, pred)),
        "MAE": mean_absolute_error(y, pred),
    }

results = pd.DataFrame({
    "train": evaluate(model, X_train, y_train),
    "test":  evaluate(model, X_test, y_test),
}).T
results

In [ ]:
pred_test = model.predict(X_test)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].scatter(y_test, pred_test, s=5, alpha=0.4)
lims = [y_test.min(), y_test.max()]
ax[0].plot(lims, lims, "r--")
ax[0].set(xlabel="True ($100k)", ylabel="Predicted ($100k)", title="Predicted vs True (test)")

ax[1].hist(y_test - pred_test, bins=50)
ax[1].set(xlabel="Residual ($100k)", title="Residuals (test)")
plt.tight_layout()
plt.show()

## 5. Checking overfitting
Three checks:
1. **Train vs test score gap** (above) – a big gap means overfitting.
2. **Cross-validation** with train and validation scores.
3. **Learning curve** – and a comparison with a deliberately complex model (polynomial degree 1 → 4) to show what overfitting looks like.

In [ ]:
cv = cross_validate(
    make_model(LinearRegression()), X_train, y_train,
    cv=5, scoring="r2", return_train_score=True
)
print(f"CV train R2: {cv['train_score'].mean():.3f} ± {cv['train_score'].std():.3f}")
print(f"CV valid R2: {cv['test_score'].mean():.3f} ± {cv['test_score'].std():.3f}")

In [ ]:
sizes, train_scores, val_scores = learning_curve(
    make_model(LinearRegression()), X_train, y_train,
    train_sizes=np.linspace(0.1, 1.0, 8), cv=5, scoring="r2", random_state=RANDOM_STATE
)
plt.plot(sizes, train_scores.mean(axis=1), "o-", label="Train")
plt.plot(sizes, val_scores.mean(axis=1), "o-", label="Validation")
plt.xlabel("Training set size"); plt.ylabel("R²")
plt.title("Learning curve (linear model)")
plt.legend(); plt.grid(alpha=0.3); plt.show()

In [ ]:
# Model complexity vs. overfitting (polynomial degree)
rows = []
for degree in [1, 2, 3, 4]:
    pipe = Pipeline([
        ("clip", Winsorizer()),
        ("scale", StandardScaler()),
        ("poly", PolynomialFeatures(degree, include_bias=False)),
        ("scale2", StandardScaler()),
        ("reg", Ridge(alpha=1e-3)),
    ])
    pipe.fit(X_train, y_train)
    rows.append({
        "degree": degree,
        "train_R2": r2_score(y_train, pipe.predict(X_train)),
        "test_R2": r2_score(y_test, pipe.predict(X_test)),
    })
comp = pd.DataFrame(rows).set_index("degree")
display(comp)
comp.plot(marker="o", title="Train vs Test R² by model complexity")
plt.ylim(comp.min().min() - 0.05, 1); plt.grid(alpha=0.3); plt.show()

**How to read it**
- Linear model: train and test R² are close → **no significant overfitting**. If both are low, the model is *underfitting* (too simple), not overfitting.
- Higher polynomial degrees: train R² goes up while test R² stalls or drops → classic overfitting.
- To reduce overfitting if it appears: use `Ridge`/`Lasso` regularization, fewer features, or more data.

In [ ]:
# Final check on the held-out test set
final = pd.DataFrame({"train": evaluate(model, X_train, y_train),
                      "test": evaluate(model, X_test, y_test)}).T
gap = final.loc["train", "R2"] - final.loc["test", "R2"]
print(f"R² gap (train - test): {gap:.4f}")
print("➜ Overfitting is NOT a problem." if gap < 0.05 else "➜ Possible overfitting, try regularization.")

## 6. Small user interface
Enter the 8 variables of a house and click **Predict**.

| Variable | Meaning |
|---|---|
| MedInc | Median income in the block (in $10,000) |
| HouseAge | Median house age (years) |
| AveRooms | Average number of rooms per household |
| AveBedrms | Average number of bedrooms per household |
| Population | Block population |
| AveOccup | Average number of occupants per household |
| Latitude / Longitude | Location in California |

> Requires `ipywidgets` (works in Jupyter Notebook / JupyterLab / VS Code).

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# Retrain the final model on ALL cleaned data for the UI (optional but common)
final_model = make_model(LinearRegression()).fit(X, y)

descriptions = {
    "MedInc": "Median income ($10k)",
    "HouseAge": "House age (years)",
    "AveRooms": "Avg rooms",
    "AveBedrms": "Avg bedrooms",
    "Population": "Population",
    "AveOccup": "Avg occupants",
    "Latitude": "Latitude",
    "Longitude": "Longitude",
}

inputs = {}
for col in FEATURES:
    lo, hi = float(X[col].quantile(0.01)), float(X[col].quantile(0.99))
    inputs[col] = widgets.FloatText(
        value=round(float(X[col].median()), 2),
        description=descriptions[col],
        style={"description_width": "170px"},
        layout=widgets.Layout(width="330px"),
        tooltip=f"Typical range: {lo:.2f} to {hi:.2f}",
    )

button = widgets.Button(description="Predict price", button_style="success", icon="home")
output = widgets.Output()

def on_click(_):
    with output:
        clear_output()
        row = pd.DataFrame([{c: w.value for c, w in inputs.items()}])[FEATURES]
        price = final_model.predict(row)[0] * 100_000
        # Warn if inputs are outside the range seen during training
        warn = [c for c in FEATURES if not (X[c].min() <= row[c].iloc[0] <= X[c].max())]
        print(f"🏠 Estimated house price: ${price:,.0f}")
        if warn:
            print("⚠️ Outside training range (prediction less reliable):", ", ".join(warn))

button.on_click(on_click)
display(widgets.VBox([widgets.HTML("<h3>California House Price Predictor</h3>"),
                      *inputs.values(), button, output]))